# Basic analysis of `papers_filtered.parquet` / `references_subset.parquet`

Quick look at the ogbn-arxiv + expanded-references subset produced by `Download_script/filter_ogbn_references.py`: shapes, columns, and basic node/edge counts.

In [6]:
import pandas as pd

PAPERS_PATH = "processed/papers_filtered.parquet"
REFERENCES_PATH = "processed/references_subset.parquet"

papers = pd.read_parquet(PAPERS_PATH)
references = pd.read_parquet(REFERENCES_PATH)

## `papers_filtered.parquet`

In [7]:
print("shape:", papers.shape)
print("columns:", list(papers.columns))
papers.dtypes

shape: (314224, 5)
columns: ['mag_id', 'corpusid', 'abstract', 'title', 'year']


mag_id        int64
corpusid      int64
abstract     object
title        object
year        float64
dtype: object

In [8]:
papers.head()

,mag_id,corpusid,abstract,title,year
0,2294267086,6341238,It is well known that when $f(v)$ is a constan...,A Characterization for the Existence of Connec...,2016.0
1,2591100698,17981075,Background and purpose The heart is a complex ...,A cardiac contouring atlas for radiotherapy,2017.0
2,2896825822,52984854,Word embeddings are undoubtedly very useful co...,Word Embeddings from Large-Scale Greek Web Con...,2018.0
3,2947622387,170079093,Deep Neural Network (DNN) trained by the gradi...,Robust Sparse Regularization: Simultaneously O...,2019.0
4,37526647,13876865,Design issues of a spontaneous speech corpus i...,Spontaneous Speech Corpus of Japanese,2000.0


In [9]:
print("rows:", len(papers))
print("distinct mag_id (nodes):", papers["mag_id"].nunique())
print("duplicate mag_id rows:", len(papers) - papers["mag_id"].nunique())
print("null counts:")
papers.isna().sum()

rows: 314224
distinct mag_id (nodes): 314224
duplicate mag_id rows: 0
null counts:


mag_id       0
corpusid     0
abstract     0
title        0
year        28
dtype: int64

## `references_subset.parquet`

In [10]:
print("shape:", references.shape)
print("columns:", list(references.columns))
references.dtypes

shape: (2986960, 2)
columns: ['citing_mag_id', 'cited_mag_id']


citing_mag_id    int64
cited_mag_id     int64
dtype: object

In [11]:
references.head()

,citing_mag_id,cited_mag_id
0,2951784689,2083195487
1,2972646545,2950468330
2,2890595896,2963200020
3,3022652760,2584377191
4,3004276397,2462155071


In [12]:
print("rows (edges):", len(references))
print("duplicate edges:", len(references) - len(references.drop_duplicates()))
print("self-loops (citing == cited):", (references["citing_mag_id"] == references["cited_mag_id"]).sum())
print("null counts:")
references.isna().sum()

rows (edges): 2986960
duplicate edges: 0
self-loops (citing == cited): 20
null counts:


citing_mag_id    0
cited_mag_id     0
dtype: int64

## Node/edge consistency between the two files

In [8]:
paper_ids = set(papers["mag_id"])
edge_ids = set(references["citing_mag_id"]) | set(references["cited_mag_id"])

print("nodes in papers_filtered:", len(paper_ids))
print("distinct node ids appearing in references_subset:", len(edge_ids))
print("edge endpoints missing from papers_filtered (should be 0):", len(edge_ids - paper_ids))
print("papers_filtered nodes with no edges at all:", len(paper_ids - edge_ids))

nodes in papers_filtered: 314224
distinct node ids appearing in references_subset: 313607
edge endpoints missing from papers_filtered (should be 0): 0


papers_filtered nodes with no edges at all:

 617


In [9]:
out_degree = references.groupby("citing_mag_id").size()
in_degree = references.groupby("cited_mag_id").size()

print("out-degree (references made) stats:")
print(out_degree.describe())
print("\nin-degree (citations received) stats:")
print(in_degree.describe())

out-degree (references made) stats:
count    280685.000000
mean         10.641680
std          12.223613
min           1.000000
25%           3.000000
50%           7.000000
75%          14.000000
max         738.000000
dtype: float64

in-degree (citations received) stats:
count    277703.000000
mean         10.755952
std          73.660489
min           1.000000
25%           1.000000
50%           3.000000
75%           8.000000
max       16504.000000
dtype: float64
